In [5]:
!pip install yfinance pandas numpy plotly requests

In [7]:
import pandas as pd
import yfinance as yf

# Fetching screening data. Hardcoded universe — S&P 100 + key ETFs + REITs
universe = [
    # S&P 100 large caps
    'AAPL','MSFT','GOOGL','AMZN','NVDA','META','TSLA','BRK-B','JPM','JNJ',
    'V','UNH','XOM','WMT','MA','PG','HD','CVX','MRK','ABBV',
    'PEP','KO','AVGO','COST','TMO','MCD','ACN','BAC','LIN','ABT',
    'CRM','DHR','NEE','CMCSA','NFLX','TXN','PM','RTX','AMGN','UNP',
    'HON','LOW','QCOM','CAT','SPGI','GS','BLK','AXP','SBUX','PLD',
    'DE','MMM','IBM','GILD','BA','ISRG','SYK','ADI','REGN','VRTX',
    'MDLZ','CI','CB','SO','DUK','MO','ZTS','EOG','SLB','PXD',
    'COP','OXY','MPC','PSX','VLO','WFC','USB','TFC','PNC','SCHW',
    'MS','BK','COF','AIG','PRU','MET','ALL','TRV','AFL','HIG',
    # ETFs
    'SPY','QQQ','IWM','VTI','VEA','VWO',
    'XLF','XLK','XLE','XLV','XLI','XLP',
    'AGG','TLT','HYG','LQD',
    # REITs
    'VNQ','O','AMT','WELL','SPG','PSA',
    # Commodities
    'GLD','SLV','USO',
]

universe = list(set(universe))
print(f"Universe size: {len(universe)} tickers")

Universe size: 115 tickers


In [8]:
import numpy as np

print("Fetching data... this will take ~1-2 minutes")

# Fetch 1 year of price data for momentum
prices = yf.download(universe, period='1y', auto_adjust=True, progress=False)['Close']
prices = prices.dropna(axis=1, thresh=int(len(prices)*0.8))  # drop tickers with >20% missing data

# Fetch fundamental data
records = []
for ticker in prices.columns:
    try:
        info = yf.Ticker(ticker).info

        # Valuation
        pe     = info.get('trailingPE', None)
        pb     = info.get('priceToBook', None)
        ev_eb  = info.get('enterpriseToEbitda', None)

        # Quality
        roe    = info.get('returnOnEquity', None)
        margin = info.get('profitMargins', None)
        de     = info.get('debtToEquity', None)

        # Momentum (52-week return)
        if ticker in prices.columns and len(prices[ticker].dropna()) > 50:
            mom = (prices[ticker].iloc[-1] / prices[ticker].iloc[0] - 1) * 100
        else:
            mom = None

        records.append({
            'ticker': ticker, 'pe': pe, 'pb': pb, 'ev_ebitda': ev_eb,
            'roe': roe, 'margin': margin, 'de': de, 'momentum': mom
        })
    except:
        pass

df = pd.DataFrame(records).set_index('ticker')
print(f"\nData fetched for {len(df)} tickers")
print(df.head())

Fetching data... this will take ~1-2 minutes



1 Failed download:
['PXD']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')



Data fetched for 114 tickers
                pe         pb  ev_ebitda      roe   margin      de   momentum
ticker                                                                       
AAPL     35.833940  40.720387     26.871  1.41471  0.27152  79.548  48.997403
ABBV    109.643906 -59.668170     15.410      NaN  0.05786     NaN  20.617709
ABT      25.112045   2.999431     15.565  0.12334  0.13905  64.773 -32.956098
ACN      13.719902   3.300106      8.259  0.24763  0.10606  25.469 -46.630130
ADI      61.420270   5.949359     31.960  0.09639  0.26008  25.810  79.933790


In [9]:
# Score and Rank 
def rank_score(series, ascending=True):
    """Rank a series 0-100, handling NaN gracefully."""
    return series.rank(ascending=ascending, pct=True) * 100

scored = df.copy()

# --- Valuation score (lower is better) ---
scored['val_pe']    = rank_score(df['pe'], ascending=True)
scored['val_pb']    = rank_score(df['pb'], ascending=True)
scored['val_eveb']  = rank_score(df['ev_ebitda'], ascending=True)
scored['val_score'] = scored[['val_pe','val_pb','val_eveb']].mean(axis=1)

# --- Quality score (higher is better) ---
scored['q_roe']     = rank_score(df['roe'], ascending=False)
scored['q_margin']  = rank_score(df['margin'], ascending=False)
scored['q_de']      = rank_score(df['de'], ascending=True)   # lower debt = better
scored['q_score']   = scored[['q_roe','q_margin','q_de']].mean(axis=1)

# --- Momentum score (higher is better) ---
scored['mom_score'] = rank_score(df['momentum'], ascending=False)

# --- Composite score (equal weight: 33/33/33) ---
scored['composite'] = (
    scored['val_score'] * 0.33 +
    scored['q_score']   * 0.33 +
    scored['mom_score'] * 0.34
)

# --- Final leaderboard ---
cols = ['composite','val_score','q_score','mom_score','pe','pb','roe','margin','momentum']
leaderboard = scored[cols].sort_values('composite', ascending=False).round(1)

print("TOP 20 OPPORTUNITIES")
print("=" * 80)
print(leaderboard.head(20).to_string())

TOP 20 OPPORTUNITIES
        composite  val_score  q_score  mom_score     pe    pb  roe  margin  momentum
ticker                                                                              
COST         76.4       93.2     56.4       79.6   49.1  26.2  0.3     0.0      -2.1
COF          75.6       52.1     89.3       85.0   56.0   1.0  0.0     0.1      -6.6
ISRG         74.9       88.4     41.0       94.7   50.1   8.4  0.2     0.3     -19.5
MDLZ         74.5       64.0     77.0       82.3   31.4   3.2  0.1     0.1      -3.9
SYK          74.4       70.7     58.0       93.8   35.3   5.2  0.2     0.1     -18.9
AMT          72.7       78.1     52.7       86.7   30.5  25.0  0.3     0.3      -9.8
SBUX         71.0       61.8     91.5       60.2   78.1 -13.8  NaN     0.0      11.3
HON          70.6       75.6     66.4       69.9   35.0  10.2  0.2     0.1       5.1
DHR          70.4       62.1     60.1       88.5   35.0   2.4  0.1     0.1     -11.2
ABT          69.4       51.1     59.8       

In [10]:
from datetime import datetime

# Clean display — top 15 with flags
watchlist = leaderboard.head(15).copy()

# Add flags for interesting situations
watchlist['flags'] = ''
watchlist.loc[watchlist['momentum'] < -20, 'flags'] += '📉 oversold  '
watchlist.loc[watchlist['momentum'] > 20,  'flags'] += '📈 momentum  '
watchlist.loc[watchlist['pe'] > 100,        'flags'] += '⚠️ high PE  '
watchlist.loc[watchlist['pb'] < 0,          'flags'] += '⚠️ neg book  '
watchlist.loc[watchlist['roe'] > 0.3,       'flags'] += '⭐ high ROE  '
watchlist.loc[watchlist['margin'] > 0.25,   'flags'] += '⭐ fat margin  '

# Format for readability
display = pd.DataFrame({
    'Score'    : watchlist['composite'],
    'Valuation': watchlist['val_score'],
    'Quality'  : watchlist['q_score'],
    'Momentum' : watchlist['mom_score'],
    'P/E'      : watchlist['pe'],
    'ROE'      : (watchlist['roe'] * 100).round(1).astype(str) + '%',
    'Margin'   : (watchlist['margin'] * 100).round(1).astype(str) + '%',
    '1Y Ret'   : watchlist['momentum'].round(1).astype(str) + '%',
    'Flags'    : watchlist['flags'],
})

print(f"\n📊 DAILY WATCHLIST — {datetime.today().strftime('%B %d, %Y')}")
print("=" * 100)
print(display.to_string())
print("\n💡 Tip: Re-run this notebook each morning for a fresh ranking.")


📊 DAILY WATCHLIST — June 11, 2026
        Score  Valuation  Quality  Momentum   P/E     ROE Margin  1Y Ret                                   Flags
ticker                                                                                                          
COST     76.4       93.2     56.4      79.6  49.1   30.0%   0.0%   -2.1%                                        
COF      75.6       52.1     89.3      85.0  56.0    0.0%  10.0%   -6.6%                                        
ISRG     74.9       88.4     41.0      94.7  50.1   20.0%  30.0%  -19.5%                          ⭐ fat margin  
MDLZ     74.5       64.0     77.0      82.3  31.4   10.0%  10.0%   -3.9%                                        
SYK      74.4       70.7     58.0      93.8  35.3   20.0%  10.0%  -18.9%                                        
AMT      72.7       78.1     52.7      86.7  30.5   30.0%  30.0%   -9.8%                          ⭐ fat margin  
SBUX     71.0       61.8     91.5      60.2  78.1    nan%   0

In [11]:
script = '''
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

universe = [
    'AAPL','MSFT','GOOGL','AMZN','NVDA','META','TSLA','BRK-B','JPM','JNJ',
    'V','UNH','XOM','WMT','MA','PG','HD','CVX','MRK','ABBV',
    'PEP','KO','AVGO','COST','TMO','MCD','ACN','BAC','LIN','ABT',
    'CRM','DHR','NEE','CMCSA','NFLX','TXN','PM','RTX','AMGN','UNP',
    'HON','LOW','QCOM','CAT','SPGI','GS','BLK','AXP','SBUX','PLD',
    'DE','MMM','IBM','GILD','BA','ISRG','SYK','ADI','REGN','VRTX',
    'MDLZ','CI','CB','SO','DUK','MO','ZTS','EOG','SLB',
    'COP','OXY','MPC','PSX','VLO','WFC','USB','TFC','PNC','SCHW',
    'MS','BK','COF','AIG','PRU','MET','ALL','TRV','AFL','HIG',
    'SPY','QQQ','IWM','VTI','VEA','VWO',
    'XLF','XLK','XLE','XLV','XLI','XLP',
    'AGG','TLT','HYG','LQD',
    'VNQ','O','AMT','WELL','SPG','PSA',
    'GLD','SLV','USO',
]

print("Fetching data...")
prices = yf.download(universe, period='1y', auto_adjust=True, progress=False)['Close']
prices = prices.dropna(axis=1, thresh=int(len(prices)*0.8))

records = []
for ticker in prices.columns:
    try:
        info = yf.Ticker(ticker).info
        pe     = info.get('trailingPE', None)
        pb     = info.get('priceToBook', None)
        ev_eb  = info.get('enterpriseToEbitda', None)
        roe    = info.get('returnOnEquity', None)
        margin = info.get('profitMargins', None)
        de     = info.get('debtToEquity', None)
        mom    = (prices[ticker].iloc[-1] / prices[ticker].iloc[0] - 1) * 100 if len(prices[ticker].dropna()) > 50 else None
        records.append({'ticker': ticker, 'pe': pe, 'pb': pb, 'ev_ebitda': ev_eb,
                        'roe': roe, 'margin': margin, 'de': de, 'momentum': mom})
    except:
        pass

df = pd.DataFrame(records).set_index('ticker')

def rank_score(series, ascending=True):
    return series.rank(ascending=ascending, pct=True) * 100

scored = df.copy()
scored['val_score'] = rank_score(df['pe'], ascending=True) * 0.33 + rank_score(df['pb'], ascending=True) * 0.33 + rank_score(df['ev_ebitda'], ascending=True) * 0.34
scored['q_score']   = rank_score(df['roe'], ascending=False) * 0.33 + rank_score(df['margin'], ascending=False) * 0.33 + rank_score(df['de'], ascending=True) * 0.34
scored['mom_score'] = rank_score(df['momentum'], ascending=False)
scored['composite'] = scored['val_score'] * 0.33 + scored['q_score'] * 0.33 + scored['mom_score'] * 0.34

leaderboard = scored[['composite','val_score','q_score','mom_score','pe','pb','roe','margin','momentum']].sort_values('composite', ascending=False).round(1)
watchlist = leaderboard.head(15).copy()

watchlist['flags'] = ''
watchlist.loc[watchlist['momentum'] < -20, 'flags'] += '📉 oversold  '
watchlist.loc[watchlist['momentum'] > 20,  'flags'] += '📈 momentum  '
watchlist.loc[watchlist['pe'] > 100,        'flags'] += '⚠️ high PE  '
watchlist.loc[watchlist['pb'] < 0,          'flags'] += '⚠️ neg book  '
watchlist.loc[watchlist['roe'] > 0.3,       'flags'] += '⭐ high ROE  '
watchlist.loc[watchlist['margin'] > 0.25,   'flags'] += '⭐ fat margin  '

display = pd.DataFrame({
    'Score'    : watchlist['composite'],
    'Valuation': watchlist['val_score'],
    'Quality'  : watchlist['q_score'],
    'Momentum' : watchlist['mom_score'],
    'P/E'      : watchlist['pe'],
    'ROE'      : (watchlist['roe'] * 100).round(1).astype(str) + '%',
    'Margin'   : (watchlist['margin'] * 100).round(1).astype(str) + '%',
    '1Y Ret'   : watchlist['momentum'].round(1).astype(str) + '%',
    'Flags'    : watchlist['flags'],
})

print(f"\\n📊 DAILY WATCHLIST — {datetime.today().strftime('%B %d, %Y')}")
print("=" * 100)
print(display.to_string())
print("\\n💡 Tip: Re-run this script each morning for a fresh ranking.")
'''

with open('daily_screener.py', 'w') as f:
    f.write(script)

print("✅ Saved as daily_screener.py")
print("Every morning, just run: python daily_screener.py")

✅ Saved as daily_screener.py
Every morning, just run: python daily_screener.py


What you built:
A quantitative stock screener that runs every morning, pulls live market data, and ranks 114 stocks, ETFs, and REITs using three factors every institutional investor cares about — valuation, quality, and momentum. It spits out a ranked shortlist with flags so you know exactly where to focus your research time.

Why it's actually useful
The problem most investors have isn't analysis — it's knowing what to analyze. You could spend hours on a DCF for a name that's expensive, low quality, and in a downtrend. This screener solves that by filtering the universe down to 15 names worth your attention before you open a single 10-K.
Think of it as your first filter in a two-step process — the screener surfaces candidates, your CFA-trained brain does the deep work on the shortlist.

How to showcase it
There are three audiences and a different angle for each:
For buy-side roles (asset managers, hedge funds, family offices) — run it every morning for 2-3 weeks, pick 2-3 names that score consistently high, write a short investment memo on each. In the interview say: "I built a factor-based screener that I run daily. Here are three ideas it surfaced and here's my thesis on each." That's a candidate who walks in with live, current ideas — extremely rare.
For investment analyst roles (like your current one at FIRM) — frame it as a process improvement. You built a systematic way to monitor a universe of names across valuation, quality, and momentum. It's the kind of tool an IA at an advisory firm would use to support manager due diligence and asset allocation decisions.
For PE/IB roles — emphasize the quality and momentum factors. PE firms care deeply about margin profiles, ROE, and business momentum. Show them the flags logic and explain how it helps triage acquisition targets or public comps quickly.

The one thing that makes this stand out
Most finance candidates who know Python build something in a tutorial and never use it again. You built something you actually open every morning. That authenticity comes through in interviews — and the fact that you can point to a specific idea it surfaced, with a real thesis behind it, is what separates you from everyone else.